# РК 1. EDA-анализ данных об SQL-инъекциях

Перед запуском выполните client.py, чтобы получить sql_injections_merged.csv.

In [ ]:
from collections import Counter
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid")

CSV_PATH = Path("sql_injections_merged.csv")
if not CSV_PATH.exists():
    raise FileNotFoundError(
        "Файл sql_injections_merged.csv не найден. Сначала выполните: python client.py"
    )

df = pd.read_csv(CSV_PATH)
print(f"Загружено строк: {len(df):,}")
print(f"Столбцов: {len(df.columns)}")
display(df.head())

In [ ]:
def split_values(series, pattern=r"[,;#]", lower=False):
    values = []
    for value in series.dropna().astype(str):
        for part in re.split(pattern, value):
            token = part.strip()
            if lower:
                token = token.lower()
            if token:
                values.append(token)
    return values

def plot_counter(counter, top_n, title):
    top = pd.Series(dict(counter.most_common(top_n)), dtype="int64")
    if top.empty:
        print("Нет данных для построения графика")
        return top
    plt.figure(figsize=(10, max(5, top_n * 0.32)))
    top.sort_values().plot(kind="barh")
    plt.title(title)
    plt.xlabel("Количество")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()
    return top

## 1. Общая информация: размер, типы, пропуски и дубликаты

In [ ]:
print("Размер DataFrame:", df.shape)
display(df.dtypes.to_frame("dtype"))

missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
}).sort_values("missing_count", ascending=False)
display(missing)

plt.figure(figsize=(12, 6))
missing["missing_count"].sort_values().plot(kind="barh")
plt.title("Количество пропусков по столбцам")
plt.xlabel("Количество пропусков")
plt.tight_layout()
plt.show()

duplicates = int(df.duplicated().sum())
max_col = missing.index[0]
max_count = int(missing.iloc[0]["missing_count"])
if max_count:
    print(f"Вывод: пропуски есть. Больше всего в '{max_col}': {max_count} ({missing.iloc[0]['missing_percent']:.2f}%).")
else:
    print("Вывод: пропусков нет.")
print(f"Полных дубликатов строк: {duplicates}.")

## 2. Анализ Category

In [ ]:
category_counts = df["Category"].fillna("<NULL>").value_counts()
category_share = (df["Category"].fillna("<NULL>").value_counts(normalize=True) * 100).round(2)
display(pd.DataFrame({"count": category_counts, "share_%": category_share}).head(10))

plt.figure(figsize=(10, 6))
category_counts.head(10).sort_values().plot(kind="barh")
plt.title("Топ-10 категорий")
plt.tight_layout()
plt.show()

pie = category_counts.head(8).copy()
other = int(category_counts.iloc[8:].sum())
if other:
    pie.loc["Прочие"] = other
plt.figure(figsize=(8, 8))
plt.pie(pie.values, labels=pie.index, autopct="%1.1f%%", startangle=90)
plt.title("Category: топ-8 + прочие")
plt.tight_layout()
plt.show()

print(f"Вывод: доминирует '{category_counts.index[0]}' ({category_share.iloc[0]:.2f}%). Уникальных категорий: {df['Category'].nunique(dropna=True)}.")

## 3. Анализ Attack Type

In [ ]:
attack_counts = df["Attack Type"].fillna("<NULL>").value_counts()
display(attack_counts.head(10).to_frame("count"))

plt.figure(figsize=(10, 6))
attack_counts.head(10).sort_values().plot(kind="barh")
plt.title("Топ-10 типов атак")
plt.tight_layout()
plt.show()

unique_attack_types = df["Attack Type"].nunique(dropna=True)
print(f"Вывод: чаще всего встречается '{attack_counts.index[0]}'. Уникальных типов атак: {unique_attack_types}.")

## 4. Анализ Tools Used

In [ ]:
tools = split_values(df["Tools Used"], pattern=r"[,;#]")
tools_counter = Counter(tools)
top_tools = plot_counter(tools_counter, 15, "Топ-15 инструментов")
display(top_tools.sort_values(ascending=False).to_frame("count"))
names = [name for name, _ in tools_counter.most_common(3)]
print("Вывод: самые популярные инструменты: " + (", ".join(names) if names else "не выделены") + ".")

## 5. Анализ MITRE Technique

In [ ]:
mitre_pattern = r"T\d{4}(?:\.\d{3})?"
mitre_all = []
for value in df["MITRE Technique"].dropna().astype(str):
    mitre_all.extend(re.findall(mitre_pattern, value))

mitre_counter = Counter(mitre_all)
top_mitre = plot_counter(mitre_counter, 10, "Топ-10 MITRE ATT&CK техник")
display(top_mitre.sort_values(ascending=False).to_frame("count"))
print("Вывод: доминируют техники: " + (", ".join(name for name, _ in mitre_counter.most_common(3)) if mitre_counter else "не выделены") + ".")

## 6. Анализ Tags

In [ ]:
tags = split_values(df["Tags"], pattern=r"[,;#]", lower=True)
tags_counter = Counter(tags)
top_tags = plot_counter(tags_counter, 20, "Топ-20 тегов")
display(top_tags.sort_values(ascending=False).to_frame("count"))
print("Вывод: наиболее частые теги: " + (", ".join(name for name, _ in tags_counter.most_common(5)) if tags_counter else "не выделены") + ".")

## 7. Анализ Source

In [ ]:
sources = split_values(df["Source"], pattern=r"[,;]")
sources_counter = Counter(sources)
top_sources = plot_counter(sources_counter, 15, "Топ-15 источников")
display(top_sources.sort_values(ascending=False).to_frame("count"))
print("Вывод: наиболее частые источники: " + (", ".join(name for name, _ in sources_counter.most_common(5)) if sources_counter else "не выделены") + ". Частота показывает вклад источников в состав датасета, а их авторитетность оценивается отдельно.")

## 8. Category × Attack Type

In [ ]:
top_categories = df["Category"].value_counts().head(10).index
top_attacks = df["Attack Type"].value_counts().head(10).index
subset = df[df["Category"].isin(top_categories) & df["Attack Type"].isin(top_attacks)]
ct_cat_attack = pd.crosstab(subset["Category"], subset["Attack Type"])
display(ct_cat_attack)

plt.figure(figsize=(14, 8))
sns.heatmap(ct_cat_attack, annot=True, fmt="d", cmap="Blues")
plt.title("Category × Attack Type: абсолютные значения")
plt.tight_layout()
plt.show()

row_share = ct_cat_attack.div(ct_cat_attack.sum(axis=1).replace(0, np.nan), axis=0)
plt.figure(figsize=(14, 8))
sns.heatmap(row_share, annot=True, fmt=".2f", cmap="YlOrRd")
plt.title("Category × Attack Type: доли по строке")
plt.tight_layout()
plt.show()

if not ct_cat_attack.empty:
    strongest = ct_cat_attack.stack().sort_values(ascending=False)
    pair = strongest.index[0]
    print(f"Вывод: самая выраженная пара — '{pair[0]}' → '{pair[1]}' ({int(strongest.iloc[0])} записей).")

## 9. Attack Type × MITRE

In [ ]:
def first_mitre(value):
    if pd.isna(value):
        return np.nan
    found = re.findall(mitre_pattern, str(value))
    return found[0] if found else np.nan

df["mitre_main"] = df["MITRE Technique"].apply(first_mitre)
top8_attack = df["Attack Type"].value_counts().head(8).index
top10_mitre = df["mitre_main"].value_counts().head(10).index
subset = df[df["Attack Type"].isin(top8_attack) & df["mitre_main"].isin(top10_mitre)]
ct_attack_mitre = pd.crosstab(subset["Attack Type"], subset["mitre_main"])
display(ct_attack_mitre)

plt.figure(figsize=(14, 7))
sns.heatmap(ct_attack_mitre, annot=True, fmt="d", cmap="Purples")
plt.title("Attack Type × MITRE Technique")
plt.tight_layout()
plt.show()

if not ct_attack_mitre.empty:
    strongest = ct_attack_mitre.stack().sort_values(ascending=False)
    pair = strongest.index[0]
    print(f"Вывод: наиболее выраженная связь — '{pair[0]}' с {pair[1]} ({int(strongest.iloc[0])} записей).")

## 10. Длины текстов

In [ ]:
text_columns = {
    "Scenario Description": "len_scenario_description",
    "Attack Steps": "len_attack_steps",
    "Solution": "len_solution",
}
for source_col, length_col in text_columns.items():
    df[length_col] = df[source_col].fillna("").astype(str).str.len()

length_cols = list(text_columns.values())
display(df[length_cols].describe().round(1))

for source_col, length_col in text_columns.items():
    plt.figure(figsize=(10, 5))
    sns.histplot(x=df[length_col], bins=40, kde=True, color="steelblue")
    plt.title(f"Распределение длины: {source_col}")
    plt.xlabel("Количество символов")
    plt.tight_layout()
    plt.show()

means = df[length_cols].mean().sort_values(ascending=False)
longest = means.index[0]
source_longest = {v: k for k, v in text_columns.items()}[longest]
q1, q3 = df[longest].quantile([0.25, 0.75])
iqr = q3 - q1
outliers = int((df[longest] > q3 + 1.5 * iqr).sum())
print(f"Вывод: в среднем самое длинное поле — '{source_longest}' ({means.iloc[0]:.1f} символа). Верхних выбросов по 1.5×IQR: {outliers}.")

## 11. Частые слова в Solution

In [ ]:
STOP_WORDS = {
    "the","a","an","and","or","of","to","in","for","on","with","by","from","is","are","be",
    "as","at","that","this","these","those","it","its","into","can","should","would","could",
    "such","using","use","used","ensure","all","your","their","they","you","we","not","if",
    "when","where","which","who","application","applications","system","systems","data"
}
solution_words = []
for text in df["Solution"].dropna().astype(str):
    tokens = re.findall(r"[A-Za-z][A-Za-z\-]+", text.lower())
    solution_words.extend(token for token in tokens if token not in STOP_WORDS and len(token) > 2)

solution_counter = Counter(solution_words)
top_solution = plot_counter(solution_counter, 20, "Топ-20 слов в Solution")
display(top_solution.sort_values(ascending=False).to_frame("count"))
print("Вывод: наиболее частая лексика рекомендаций: " + (", ".join(name for name, _ in solution_counter.most_common(8)) if solution_counter else "не выделена") + ".")

## 12. Финальная сводка

In [ ]:
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "missing": df.isna().sum(),
    "missing_%": (df.isna().mean() * 100).round(2),
    "unique": df.nunique(dropna=True),
})
display(summary)

## Финальные выводы

In [ ]:
main_categories = ", ".join(df["Category"].value_counts().head(3).index.astype(str))
main_attacks = ", ".join(df["Attack Type"].value_counts().head(3).index.astype(str))
main_tools = ", ".join(name for name, _ in tools_counter.most_common(5)) or "не выделены"
main_mitre = ", ".join(name for name, _ in mitre_counter.most_common(5)) or "не выделены"
main_tags = ", ".join(name for name, _ in tags_counter.most_common(5)) or "не выделены"
main_sources = ", ".join(name for name, _ in sources_counter.most_common(5)) or "не выделены"
main_solution = ", ".join(name for name, _ in solution_counter.most_common(8)) or "не выделены"
missing_total = int(df.isna().sum().sum())
missing_pct = 100 * missing_total / (df.shape[0] * df.shape[1])
duplicate_count = int(df.duplicated().sum())

print(
    f"В ходе EDA установлено, что доминирующими категориями являются {main_categories}, "
    f"а наиболее распространённые типы атак — {main_attacks}. "
    f"Наиболее популярные инструменты: {main_tools}. "
    f"Чаще всего встречаются MITRE-техники {main_mitre}. "
    f"Основные теги: {main_tags}; наиболее частые источники: {main_sources}. "
    f"В рекомендациях по защите чаще всего встречается лексика: {main_solution}. "
    f"Доля пропущенных ячеек составляет {missing_pct:.2f}%, полных дубликатов строк — {duplicate_count}."
)

category_n = df["Category"].nunique(dropna=True)
attack_n = df["Attack Type"].nunique(dropna=True)
mitre_n = len(mitre_counter)
if len(df) >= 10000 and category_n >= 5 and attack_n >= 5 and mitre_n >= 3:
    print("Общий вывод: датасет достаточно крупный и разнообразный для учебного изучения SQL-инъекций, однако его репрезентативность относительно всех реальных атак зависит от происхождения и баланса источников.")
else:
    print("Общий вывод: датасет пригоден для учебного EDA, но ограниченное разнообразие отдельных признаков требует осторожности при переносе выводов на все реальные SQL-инъекции.")